In [ ]:
import numpy as np
import pandas as pd
import datetime
import random


RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)
random.seed(RANDOM_SEED)

N = 5000

donor_types = ['Restaurant', 'Bakery', 'Caterer', 'HomeCook', 'Society', 'Event']
food_types = ['Cooked Meal', 'Fruits', 'Salads', 'Baked Goods', 'Desserts', 'Beverage']
units = ['kg', 'boxes', 'plates', 'packs']
cities = ['Bengaluru', 'Mumbai', 'Delhi', 'Hyderabad', 'Chennai', 'Pune']
packaging_types = ['Airtight', 'Open Tray', 'Covered Tray', 'Sealed Box', 'Cloth Wrap']
donor_reliability_levels = ['Low', 'Medium', 'High']


packaging_factor = {
    'Airtight': 1.0, 'Sealed Box': 0.95, 'Covered Tray': 0.85,
    'Open Tray': 0.6, 'Cloth Wrap': 0.5
}

food_type_factor = {
    'Cooked Meal': 0.9, 'Fruits': 0.95, 'Salads': 0.7,
    'Baked Goods': 0.85, 'Desserts': 0.9, 'Beverage': 0.95
}

donor_reliability_factor = {
    'High': 1.0, 'Medium': 0.9, 'Low': 0.75
}

ngo_density = {
    'Bengaluru': 0.9, 'Mumbai': 0.95, 'Delhi': 0.9,
    'Hyderabad': 0.7, 'Chennai': 0.6, 'Pune': 0.65
}


start_date = datetime.datetime(2025, 8, 1)

rows = []

for i in range(N):

    listing_id = f"L{i+1:05d}"
    donor_id = f"D{random.randint(1,800):04d}"

    donor_type = np.random.choice(donor_types, p=[0.35,0.1,0.1,0.2,0.15,0.1])
    food_type = np.random.choice(food_types, p=[0.5,0.1,0.08,0.15,0.1,0.07])
    unit = random.choice(units)
    city = np.random.choice(cities, p=[0.2,0.2,0.2,0.15,0.1,0.15])
    packaging = np.random.choice(packaging_types, p=[0.3,0.2,0.25,0.18,0.07])


    day_offset = np.random.randint(0, 30)
    hour = np.random.choice(range(24), p=[0.02]*6 + [0.08]*6 + [0.12]*6 + [0.05]*6)

    time_posted = start_date + datetime.timedelta(
        days=int(day_offset),
        hours=int(hour),
        minutes=random.randint(0,59)
    )

    time_since_cooked = max(5, int(np.random.normal(90, 60)))

    quantity = round(np.random.exponential(scale=5.0), 1)

    if unit == "plates":
        quantity = max(1, int(quantity * 2))


    distance_km = float(np.clip(np.random.normal(5, 4), 0.2, 40))

    temp_c = float(np.clip(
        np.random.normal(30 - 0.2*(time_since_cooked/60), 3),
        10, 45
    ))

    donor_reliability = np.random.choice(
        donor_reliability_levels, p=[0.25,0.45,0.3]
    )

    base_freshness = max(0, 1 - (time_since_cooked / 360))
    distance_factor = max(0.2, 1 - (distance_km / 50))

    freshness_score = np.clip(
        base_freshness
        * packaging_factor[packaging]
        * food_type_factor[food_type]
        * donor_reliability_factor[donor_reliability]
        * distance_factor
        + np.random.normal(0,0.05),
        0, 1
    )

    qty_factor = min(1, quantity / 20)

    hour_factor = 0.9 if 8 <= time_posted.hour <= 20 else 0.6

    pickup_prob = np.clip(
        0.1
        + 0.5 * freshness_score
        + 0.1 * qty_factor
        + 0.1 * donor_reliability_factor[donor_reliability]
        + 0.1 * ngo_density[city]
        + 0.05 * hour_factor
        + np.random.normal(0,0.08),
        0, 1
    )

    picked_up = int(np.random.rand() < pickup_prob)

    if np.random.rand() < 0.02:
        temp_c = None

    if np.random.rand() < 0.02:
        distance_km = None

    rows.append({
        "listing_id": listing_id,
        "donor_id": donor_id,
        "donor_type": donor_type,
        "food_type": food_type,
        "quantity": quantity,
        "unit": unit,
        "city": city,
        "packaging": packaging,
        "time_posted": time_posted.strftime("%Y-%m-%d %H:%M:%S"),
        "time_since_cooked_mins": time_since_cooked,
        "donor_reliability": donor_reliability,
        "distance_km": None if distance_km is None else round(distance_km,2),
        "ambient_temp_c": None if temp_c is None else round(temp_c,1),
        "freshness_score": round(float(freshness_score),3),
        "pickup_prob": round(float(pickup_prob),3),
        "picked_up": picked_up
    })

df = pd.DataFrame(rows)

df.to_csv("resqfood_listings.csv", index=False)

print("✅ Dataset generated successfully")
print("Rows:", len(df))
print(df.head())

Generated resqfood_listings.csv with 5000 rows
